# PART 2 - Data Analysis:
#### Objective: Estimate probabilities

### 1. Data cleaning:

In [49]:
#Categorisation of planes for DDBB:
import pandas as pd
def categorise_aircraft(flight):
    letters = ''.join(k for k in str(flight) if k.isalpha()).upper()
    number  = ''.join(k for k in str(flight) if k.isdigit())
    if letters == "QF":
        if len(number) == 3:
            return "737"
        if number.startswith("19") or number.startswith("17") or number.startswith("18"):
            return "e190"
    if letters == "QQ":
        if len(number) == 4:
            return "e190"
    if letters == "VA":
        if len(number) == 3 or len(number) == 4:
            return "737"
    return "Insufficient information"

In [51]:
data = pd.read_csv("ADL_aggregated_arrivals.csv")
data["Aircraft"] = data["Flight"].apply(categorise_aircraft)

In [53]:
status_clean = data["Status"].astype(str).str.strip().str.lower()
landed = status_clean.eq("landed")
cancelled = status_clean.str.contains("cancel")

other = ~(landed | cancelled)
print("Total database records:", len(data))
print("Landed records:", int(landed.sum()))
print("Cancelled records:", int(cancelled.sum()))
print("Other records:", int(other.sum()))


print("Landed with numerical gate:", int((landed & gated).sum()))
print("Confirmed 737/E190:", len(confirmed))
print("737:", b737_count)
print("E190:", e190_count)
print("Uncategorised landed and gated:", uncategorised_count)
print("total_count2:", total_count2)



Total database records: 4897
Landed records: 4792
Cancelled records: 64
Other records: 41
Landed with numerical gate: 3738
Confirmed 737/E190: 2635
737: 2000
E190: 635
Uncategorised landed and gated: 1103
total_count2: 3738


In [55]:
data_original = data.copy()
status_clean = data["Status"].astype(str).str.strip().str.lower()
gate_clean = data["Gate"].astype(str).str.strip()
landed = status_clean.eq("landed")
gated = gate_clean.str.isdigit()
keep = landed & gated
data_cleaned = data.loc[keep].copy()
excluded_records = data.loc[~keep].copy()

print("Original records:", len(data_original))
print("Retained landed-and-gated records:", len(data_cleaned))
print("Excluded records:", len(excluded_records))

Original records: 4897
Retained landed-and-gated records: 3738
Excluded records: 1159


In [59]:
#Confirmed:
landed = data["Status"].astype(str).str.strip().str.lower().eq("landed")
gated = data["Gate"].astype(str).str.strip().str.isdigit()
confirmed = data[landed & gated & data["Aircraft"].isin(["737", "e190"])].copy()

confirmed["hour"] = pd.to_datetime(confirmed["Estimated Time"].astype(str).str.upper(),
    format="%I:%M%p").dt.hour

days_count = confirmed["Date"].nunique()
b737_count = int((confirmed["Aircraft"] == "737").sum())
e190_count = int((confirmed["Aircraft"] == "e190").sum())
total_count = b737_count + e190_count
 
uncategorised_count = int((data.loc[landed&gated, "Aircraft"] == "Insufficient information").sum())
total_count2 = total_count + uncategorised_count

ratio_737 = (b737_count/total_count2)*100
ratio_e190 = (e190_count/total_count2)*100
concentration = ratio_737 + ratio_e190

print(f"737: {ratio_737:.2f}% from {b737_count} 737's over a total of {total_count2} landed planes.")
print(f"e190: {ratio_e190:.2f}% from {e190_count} e190's over a total of {total_count2} landed planes.")
print(f"Uncategorised aircraft: {uncategorised_count}")
print(f"Concentrating a total of {concentration:.2f}% of the landed planes")

737: 53.50% from 2000 737's over a total of 3738 landed planes.
e190: 16.99% from 635 e190's over a total of 3738 landed planes.
Uncategorised aircraft: 1103
Concentrating a total of 70.49% of the landed planes


##### 1.A. Data categories:

In [8]:
#Missed categorisation (debug my code to minimise them):
missed = data[data["Aircraft"] == "Insufficient information"].copy()
missed["letters"] = missed["Flight"].apply(lambda f: ''.join(k for k in str(f) if k.isalpha()).upper())
missed["number"]  = missed["Flight"].apply(lambda f: ''.join(k for k in str(f) if k.isdigit()))

print(f"{len(missed)} of {len(data)} flights fell through to 'Insufficient information'\n")
for letters, group in missed.groupby("letters"):
    nums = sorted(group["number"].unique())
    print(f"{letters} ({len(group)} rows): {nums}")

QF7416_count = int(confirmed["Flight"].astype(str).str.strip().str.upper().eq("QF7416").sum())

print(QF7416_count)


1783 of 4208 flights fell through to 'Insufficient information'

JQ (792 rows): ['111', '497', '499', '621', '660', '680', '689', '760', '762', '7621', '764', '766', '768', '7689', '770', '772', '774', '776', '778', '7782', '780', '782', '800', '802', '804', '807', '857', '859', '961', '963', '988']
NC (166 rows): ['217', '223', '231', '233', '237', '239', '241', '243', '245', '505', '507', '743', '745', '780', '783', '785', '787']
PFY (1 rows): ['9921']
QF (377 rows): ['1277', '1281', '1283', '1285', '2017', '2549', '2551', '2561', '2563', '2572', '2574', '2576', '2580', '2582', '2584', '2586', '2588', '6112', '6280', '6281', '6297', '7416', '7881', '7945']
VAVA (1 rows): ['720']
ZL (446 rows): ['4125', '4133', '4135', '4351', '4353', '4357', '4363', '4367', '4377', '4389', '4397', '4487', '4612', '4618', '4632', '4819', '4823', '4825', '4847', '9381', '9383', '9906', '9943', '9951']
0


### 2. Computing arrival rates:

#### Probabilities of having a specific plane, given that it's a confirmed arrival:
- P (737 | confirmed arrival) = p_737_given_arrival
- P (e190 | confirmed arrival) = p_e190_given_arrival

In [61]:
p_737_given_arrival = (b737_count/total_count)
p_e190_given_arrival = (e190_count/total_count)

print("Conditional probabilities among confirmed arrivals are as follows:")
print(f"    P(737|confirmed arrival) = {p_737_given_arrival:.4f} (from {b737_count} confirmed 737's).")
print(f"    P(e190|confirmed arrival) = {p_e190_given_arrival:.4f} (from {e190_count} confirmed e190's).")

Conditional probabilities among confirmed arrivals are as follows:
    P(737|confirmed arrival) = 0.7590 (from 2000 confirmed 737's).
    P(e190|confirmed arrival) = 0.2410 (from 635 confirmed e190's).


#### Summary of confirmed arrivals (by date, day and type of aircraft):

In [63]:
#Daily count by date:
daily = confirmed.groupby(["Date", "Aircraft"]).size().unstack(fill_value=0)
if "737" not in daily.columns:
    daily["737"] = 0
if "e190" not in daily.columns:
    daily["e190"] = 0
daily["Total"] = daily["737"] + daily["e190"]
print("\nDaily arrivals:")
print(daily)

#Means by aircraft type:
print("\nMean daily arrivals:")
print(f"    737 = {daily['737'].mean():.3f}")
print(f"    e190 = {daily['e190'].mean():.3f}")
print(f"    Total = {daily['Total'].mean():.3f}")

#Means per day:
daily_reset = daily.reset_index()
daily_reset["Date"] = pd.to_datetime(daily_reset["Date"], dayfirst=True)
daily_reset["Weekday"] = daily_reset["Date"].dt.day_name()
weekday_summary = (daily_reset.groupby("Weekday")[["737", "e190", "Total"]].mean())
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_summary = weekday_summary.reindex(weekday_order)
print("\nMean landed aircraft per day:")
for day in weekday_order:
    if day in weekday_summary.index:
        total_per_day = weekday_summary.loc[day, "Total"]
        print(f"    {day}: {total_per_day:.2f} aircraft per day")

#Conditional probabilities per day and aircraft type:
print("\nConditional probabilities per aicraft and day:")
for day in weekday_order:
    if day in weekday_summary.index:
        b737_day = weekday_summary.loc[day, "737"]
        e190_day = weekday_summary.loc[day, "e190"]
        total_day = weekday_summary.loc[day, "Total"]

        if total_day > 0:
            p_day_737 = (b737_day / total_day)
            p_day_e190 = (e190_day / total_day)

            print(f"    P(737|confirmed arrival {day}) = {p_day_737:.4f}")
            print(f"    P(e190|confirmed arrival {day}) = {p_day_e190:.4f}")


Daily arrivals:
Aircraft    737  e190  Total
Date                        
01/08/2026   28     9     37
02/08/2026   42    11     53
1/07/2026    37    14     51
10/07/2026   46    12     58
11/06/2026   43    15     58
11/07/2026   27     8     35
12/07/2026   38    10     48
13/06/2026   24     9     33
14/06/2026   39    10     49
14/07/2026   40    13     53
15/06/2026   41    17     58
15/07/2026   38    12     50
16/06/2026   33    16     49
16/07/2026   47    15     62
17/06/2026   38    16     54
17/07/2026   46    13     59
18/06/2026   40    16     56
18/07/2026   32     9     41
19/06/2026   47    15     62
19/07/2026   42    10     52
2/07/2026    42    17     59
20/06/2026   26     7     33
20/07/2026   44    13     57
21/06/2026   38    10     48
21/07/2026   36    11     47
22/06/2026   42    16     58
22/07/2026   37    12     49
23/06/2026   38    13     51
23/07/2026   45    13     58
24/06/2026   38    13     51
24/07/2026   45    13     58
25/06/2026   43    15     

#### Aircraft arrival probabilities per beat (beat = 10 minutes):
- P(E190 arrival in one beat) = p_beat_e190
- P(737 arrival in one beat) = p_beat_737
- P(no arrival in one beat) = p_beat_no_arrival


In [65]:
#In beats. W/o grouping arrivals
beat = 6 #Unit of measurement: minutes

hours_per_day = 24
mins_per_hour = 60
#days_week = 7
beats_per_day = int((hours_per_day * mins_per_hour) / beat)

p_beat_e190 = ((e190_count / days_count) / beats_per_day)
p_beat_737 = ((b737_count / days_count) / beats_per_day)
p_beat_no_arrival = 1 - (p_beat_e190 + p_beat_737)

last_date = data["Date"].iloc[-1]

print(f"As per {last_date}, considering:")
print(f"   Number of days in the database: {days_count}.")
print(f"   Hours per day: {hours_per_day}, with {mins_per_hour} minutes per hour.")
print(f"\nEach beat equals to {beat} minutes, thus every day has {beats_per_day:.1f} beats.")

print(f"\nHence, estimaded arrival probabilities per beat are:")
print(f"   P(e190 arrival in one beat) = {p_beat_e190:.4f} (from {e190_count} confirmed e190 arrivals over {days_count} days).")
print(f"   P(737 arrival in one beat) = {p_beat_737:.4f} (from {b737_count} confirmed 737 arrivals over {days_count} days).")
print(f"   P(no arrival in one beat) = {p_beat_no_arrival:.4f}")
print(f"\nNote: 3 probs should sum up to 1 (actual sum equals to = {p_beat_e190 + p_beat_737 + p_beat_no_arrival:.0f}). If otherwise, something is wrong.")


As per 02/08/2026, considering:
   Number of days in the database: 51.
   Hours per day: 24, with 60 minutes per hour.

Each beat equals to 6 minutes, thus every day has 240.0 beats.

Hence, estimaded arrival probabilities per beat are:
   P(e190 arrival in one beat) = 0.0519 (from 635 confirmed e190 arrivals over 51 days).
   P(737 arrival in one beat) = 0.1634 (from 2000 confirmed 737 arrivals over 51 days).
   P(no arrival in one beat) = 0.7847

Note: 3 probs should sum up to 1 (actual sum equals to = 1). If otherwise, something is wrong.


#### Computing probs. not considered before - batch (beat = 10 minutes):

In [15]:
print(daily["Total"].describe())
suspicious = daily[daily["Total"] < 0.6 * daily["Total"].median()]
print("Possibly incomplete days:\n", suspicious)
print(daily_reset["Weekday"].value_counts())

count    51.000000
mean     51.666667
std       8.096090
min      33.000000
25%      48.500000
50%      53.000000
75%      58.000000
max      62.000000
Name: Total, dtype: float64
Possibly incomplete days:
 Empty DataFrame
Columns: [737, e190, Total]
Index: []
Weekday
Saturday     8
Sunday       8
Thursday     8
Wednesday    7
Friday       7
Tuesday      7
Monday       6
Name: count, dtype: int64


In [18]:
batch_data = confirmed.copy()

batch_data["Date_dt"] = pd.to_datetime(batch_data["Date"], dayfirst=True)
batch_data["Time_dt"] = pd.to_datetime(batch_data["Estimated Time"].astype(str).str.upper(),format="%I:%M%p")

#Creating beats per day to check whats happening in my DDBB that im not seeing:
batch_data["beat_nr"] = (batch_data["Time_dt"].dt.hour *(mins_per_hour//beat) +
                         batch_data["Time_dt"].dt.minute//beat)
batch_data["date"] = batch_data["Date_dt"].dt.date

# Count how many planes arrived in each date and beat:
beat_counts = (batch_data.groupby(["date", "beat_nr", "Aircraft"]).size().unstack(fill_value=0))

# Double check categorisation OK:
if "737" not in beat_counts.columns:
    beat_counts["737"] = 0
if "e190" not in beat_counts.columns:
    beat_counts["e190"] = 0

beat_counts = beat_counts[["e190", "737"]]

# Include empty 10 minute beats
all_days = sorted(batch_data["date"].unique())
#
full_index = pd.MultiIndex.from_product([all_days, range(beats_per_day)],
                                        names=["date", "beat_nr"])

beat_counts = beat_counts.reindex(full_index, fill_value=0)

# Count how often each combinations happened
not_cons_probs = (beat_counts.groupby(["e190", "737"]).size().reset_index(name="frequency"))

not_cons_probs["probability"] = not_cons_probs["frequency"]/not_cons_probs["frequency"].sum()

not_cons_probs = not_cons_probs.sort_values(by="probability", ascending=False)

print("\nArrival probs per beat (beat = 6 minutes):")
print(not_cons_probs)

print("\nProbs check:")
print(f"    Sum = {not_cons_probs['probability'].sum():.6f}")



Arrival probs per beat (beat = 6 minutes):
   e190  737  frequency  probability
0     0    0       8618     0.816098
1     0    1       1237     0.117140
4     1    0        418     0.039583
2     0    2        156     0.014773
5     1    1         90     0.008523
3     0    3         15     0.001420
6     1    2         12     0.001136
8     2    0         12     0.001136
7     1    3          1     0.000095
9     3    0          1     0.000095

Probs check:
    Sum = 1.000000


In [20]:
#Checking when bays are full:
h = 5 #My assumption

#Let: state = [beat1, beat2, beat3]

state = [0, 0, 0, 0, 0]
full_bay_counter = 0 #starting counter
beats_counter = 0 #starting counter

beat_memory = [] #to save what happens in every beat

beat_counts = beat_counts.sort_index()

for i in range(len(beat_counts)):

    date_now = beat_counts.index[i][0]
    beat_now = beat_counts.index[i][1]
    e190_arrival = beat_counts.iloc[i]["e190"]
    b737_arrival = beat_counts.iloc[i]["737"]
    
    start_state = state.copy()
    state_step = [state[1], state[2], 0] #Adding a unit oof time

    pre_arrival_state = state_step.copy()
    pre_arrival_occupied = sum(pre_arrival_state)

    if pre_arrival_occupied >= h:
        pre_arrival_full = 1
    else:
        pre_arrival_full = 0
    
    accepted_e190 = 0 #starting counter
    accepted_737 = 0 #starting counter
    rejected_e190 = 0 #starting counter
    rejected_737 = 0 #starting counter
    
    flights_ordered = batch_data[(batch_data["date"] == date_now) & 
                        (batch_data["beat_nr"] == beat_now)]
    
    flights_ordered = flights_ordered.sort_values(by="Time_dt") #First in first served
    
    for j in range (len(flights_ordered)):
        current_aircraft = flights_ordered.iloc[j]["Aircraft"]
        occupied_step = sum(state_step)
        free_slot = h - occupied_step

        if free_slot > 0:        
            if current_aircraft == "e190":
                state_step[1] = state_step[1] + 1
                accepted_e190 = accepted_e190 + 1
            elif current_aircraft == "737":
                state_step[2] = state_step[2] + 1
                accepted_737 = accepted_737 + 1
        else:
            if current_aircraft == "e190":
                rejected_e190 = rejected_e190 + 1
            elif current_aircraft == "737":
                rejected_737 = rejected_737 + 1
    state = state_step.copy()
    occupied_now = sum(state)
    if occupied_now == h:
        bays_full = 1
        full_bay_counter = full_bay_counter + 1
    else:
        bays_full = 0
    beats_counter = beats_counter + 1

    beat_memory.append([date_now, beat_now, start_state, accepted_e190, accepted_737, rejected_e190, rejected_737,
                        state.copy(), occupied_now, bays_full, pre_arrival_state, pre_arrival_occupied, pre_arrival_full])

updated_bays_table = pd.DataFrame(beat_memory, columns=["date", "beat_nr", "start_state", "accepted_e190",
                                                      "accepted_737", "rejected_e190", "rejected_737",
                                                      "end_state", "occupied_bays", "bays_full", "pre_arrival_state",
                                                       "pre_arrival_occupied", "pre_arrival_full"])

full_bays_table = updated_bays_table[updated_bays_table["bays_full"] == 1]

full_bays_beats = updated_bays_table["bays_full"].sum()
total_beats = len(updated_bays_table)

prob_full_per_beat = full_bays_beats / total_beats


print(f"Overall 'Full bays' situation summary:")
print(f"   Bays are full a total of {full_bays_beats} beats over {total_beats} beats, then")
print(f"   P(Full bays per beat) = {prob_full_per_beat:.4f}, corresponding to {prob_full_per_beat*100:.2f}% of beats")


Overall 'Full bays' situation summary:
   Bays are full a total of 13 beats over 10560 beats, then
   P(Full bays per beat) = 0.0012, corresponding to 0.12% of beats


In [21]:
#Adding time column:
updated_bays_table["initial_time"] = ""
for i in range(len(updated_bays_table)):
    beat_now = updated_bays_table.iloc[i]["beat_nr"]
    start_minutes = beat_now * beat
    end_minutes = start_minutes + beat
    start_hour = start_minutes // 60
    start_minute = start_minutes % 60
    end_hour = end_minutes // 60
    end_minute = end_minutes % 60
    initial_time = f"{start_hour:02d}:{start_minute:02d}"
    updated_bays_table.loc[i, "initial_time"] = initial_time

occupied_bays_table = updated_bays_table[updated_bays_table["occupied_bays"] != 0]

display(occupied_bays_table)

,date,beat_nr,start_state,accepted_e190,accepted_737,rejected_e190,rejected_737,end_state,occupied_bays,bays_full,pre_arrival_state,pre_arrival_occupied,pre_arrival_full,initial_time
69,2026-06-11,69,"[0, 0, 0]",0,1,0,0,"[0, 0, 1]",1,0,"[0, 0, 0]",0,0,06:54
70,2026-06-11,70,"[0, 0, 1]",0,0,0,0,"[0, 1, 0]",1,0,"[0, 1, 0]",1,0,07:00
71,2026-06-11,71,"[0, 1, 0]",0,0,0,0,"[1, 0, 0]",1,0,"[1, 0, 0]",1,0,07:06
78,2026-06-11,78,"[0, 0, 0]",0,1,0,0,"[0, 0, 1]",1,0,"[0, 0, 0]",0,0,07:48
79,2026-06-11,79,"[0, 0, 1]",0,0,0,0,"[0, 1, 0]",1,0,"[0, 1, 0]",1,0,07:54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10541,2026-07-26,221,"[0, 1, 1]",0,0,0,0,"[1, 1, 0]",2,0,"[1, 1, 0]",2,0,22:06
10542,2026-07-26,222,"[1, 1, 0]",0,0,0,0,"[1, 0, 0]",1,0,"[1, 0, 0]",1,0,22:12
10544,2026-07-26,224,"[0, 0, 0]",0,1,0,0,"[0, 0, 1]",1,0,"[0, 0, 0]",0,0,22:24
10545,2026-07-26,225,"[0, 0, 1]",0,0,0,0,"[0, 1, 0]",1,0,"[0, 1, 0]",1,0,22:30


### 3. Economic estimation - inputs:

#### 3. A. Estimated arrival fee (revenue) per aircraft type:
(Note: Arrival fee is given by the "SCHEDULE OF AERONAUTICAL FEES - 01 July 2026".)
The formula to obtain the number is as follows:
- Arrival fee per aircraft = seat capacity * arrival fee per passenger



In [22]:
#Arrival fee per person:
transit_arrival_fee = 14.71 #According to "SCHEDULE OF AERONAUTICAL FEES - 01 July 2026"

#Seat capacity per aircraft:
    #e190 seat capacity:
e190_seats_capacity = 90
    #737 seat capacity:
b737_business_rows = 3
b737_business_columns = 4
b737_economy_rows = (29 - b737_business_rows)
b737_economy_columns = 6
b737_seats_capacity = (b737_business_rows * b737_business_columns) + (b737_economy_rows * b737_economy_columns)
    
#Estimated arrival fees per aircraft:
    #e190 estimated arrival fee:
e190_occupancy_level = 1 #1 = 100%. Interval between [0-1].
b737_occupancy_level = 1 #1 = 100%. Interval between [0-1].

e190_arrival_fee = (e190_seats_capacity * transit_arrival_fee) * e190_occupancy_level
    #737 estimated arrival fee:
b737_arrival_fee = (b737_seats_capacity * transit_arrival_fee) * b737_occupancy_level

print(f"e190:\n   Seat capacity: {e190_seats_capacity} passengers.\n   Estimated arrival fee: {e190_arrival_fee:.4f} AUD for a {e190_occupancy_level*100}% occupied plane.")
print(f"737:\n   Seat capacity: {b737_seats_capacity} passengers.\n   Estimated arrival fee: {b737_arrival_fee:.4f} AUD for a {b737_occupancy_level*100}% occupied plane.")

e190:
   Seat capacity: 90 passengers.
   Estimated arrival fee: 1323.9000 AUD for a 100% occupied plane.
737:
   Seat capacity: 168 passengers.
   Estimated arrival fee: 2471.2800 AUD for a 100% occupied plane.


#### 3. B. Estimated staff costs per turnaround service:
According to "Pay Guide - Airport Employees Award (MA000048) - 01 July 2026"
Assuming a standard turnaround service during the day for a 737 or E190, is assumed to consist of:
- Every worker is hired under full time/part time contracts, they are all adults, no apprenticeships:
    - Cleaning:
        - Level 2 (AUD 26.95) - Regular staff: 3 
        - Level 3 (AUD 27.54) - Supervisor: 1
    - Ramp/baggage:
        - Level 2 (AUD 26.95) - Regular staff: 4
        - Level 3 (AUD 27.54) - Supervisor: 1
    - Water/waste service:
        - Level 2 (AUD 26.95) - Regular staff: 1
    - Catering
        - Level 3 (AUD 27.54) - Regular staff: 3
    - Professional engineer:
        - Level 3 (AUD 45.41) - Regular staff: 1
     
Summarising, hourly cost per turnaround service:
- Ground staff officer - Level 2: 8
- Ground staff officer - Level 3: 5
- Professional Engineer - Level 3: 1

The general equation is given by:
- Staff cost per hour (total_cost_hour) = Total staff cost per hour  ×  Staff number
- Beat proportion (beat / mins_per_hour) = 10 (minutes) / 60 (minutes)
- Staff cost per beat (total_cost_beat) = Staff cost per hour  x  Beat proportion

##### Staff cost per beat:

In [29]:
n_ground_staff_level_2 = 8
n_ground_staff_level_3 = 5
n_engineer_level_3 = 1
cost_ground_staff_level_2 = 26.95
cost_ground_staff_level_3 = 27.54
cost_engineer_level_3 = 45.41

estimated_cost_hour = (n_ground_staff_level_2 * cost_ground_staff_level_2) + (n_ground_staff_level_3 * cost_ground_staff_level_3) + (n_engineer_level_3 * cost_engineer_level_3)
print(f"Estimated cost of a turnaround service per 1 hour is: {estimated_cost_hour:.2f} AUD.")

estimated_cost_beat = estimated_cost_hour * (beat / mins_per_hour)
print(f"Estimated cost of a turnaround service per 1 beat is: {estimated_cost_beat:.2f} AUD.")


Estimated cost of a turnaround service per 1 hour is: 398.71 AUD.
Estimated cost of a turnaround service per 1 beat is: 39.87 AUD.


##### Staff cost per plane:
From my model assumptions:
- e190_service_beats = 2 (beats to be serviced)
- b737_service_beats = 3 (beats to be serviced)

Thus, labour cost per turnaround is given by:

- e190_staff_cost = total_cost_beat x e190_service_beats
- b737_staff_cost = total_cost_beat x 737_service_beats 

In [1]:
e190_service_beats = 3
b737_service_beats = 5

e190_staff_cost = estimated_cost_beat * e190_service_beats
b737_staff_cost = estimated_cost_beat * b737_service_beats

print(f"e190:\n   Beats to be serviced: {e190_service_beats} beats.\n   Estimated staff costs: {e190_staff_cost:.4f} AUD per turnaround service.")
print(f"737:\n   Beats to be serviced: {b737_service_beats} beats.\n   Estimated staff costs: {b737_staff_cost:.4f} AUD per turnaround service.")

NameError: name 'estimated_cost_beat' is not defined

#### 3. C. Estimated net revenue* per aircraft per completed service:
Given my model assumptions, the formula for both types of aircraft are determined by:
- Net revenue = Arrival fee - staff cost

Then, for each plane:
- e190: e190_net_revenue = e190_arrival_fee - e190_staff_cost
- 737: b737_net_revenue = b737_arrival_fee - b737_staff_cost

In [33]:
e190_net_revenue = e190_arrival_fee - e190_staff_cost
b737_net_revenue = b737_arrival_fee - b737_staff_cost

print("Estimated net revenue per type of plane:")
print(f"   e190: {e190_net_revenue:.4f} AUD per each completed service.")
print(f"   b737: {b737_net_revenue:.4f} AUD per each completed service.")

Estimated net revenue per type of plane:
   e190: 1244.1580 AUD per each completed service.
   b737: 2351.6670 AUD per each completed service.


### 4. Policy selection:
- No priority: FIFS
- Priority e190
- Priority b737

In [35]:
policies_to_compare = ["fifs", "e190", "b737"]
policy_results = []

for policy in policies_to_compare:
    state = [0, 0, 0]
    total_accepted_e190 = 0
    total_accepted_737 = 0
    total_rejected_e190 = 0
    total_rejected_737 = 0
    full_bay_counter = 0
    total_beats = 0
    for i in range(len(beat_counts)):
        date_now = beat_counts.index[i][0]
        beat_now = beat_counts.index[i][1]
        state_step = [state[1], state[2], 0] #Shifting one beat
        flights_this_beat = batch_data[(batch_data["date"] == date_now) &
                                       (batch_data["beat_nr"] == beat_now)]
        flights_this_beat = flights_this_beat.sort_values(by="Time_dt")

        #Build orders s/a policy
        ordered_types = []
        if policy == "fifs":
            for j in range(len(flights_this_beat)):
                ordered_types.append(flights_this_beat.iloc[j]["Aircraft"])
        elif policy == "e190":
            for j in range(len(flights_this_beat)):
                if flights_this_beat.iloc[j]["Aircraft"] == "e190":
                    ordered_types.append("e190")
            for j in range(len(flights_this_beat)):
                if flights_this_beat.iloc[j]["Aircraft"] == "737":
                    ordered_types.append("737")
        elif policy == "b737":
            for j in range(len(flights_this_beat)):
                if flights_this_beat.iloc[j]["Aircraft"] == "737":
                    ordered_types.append("737")
            for j in range(len(flights_this_beat)):
                if flights_this_beat.iloc[j]["Aircraft"] == "e190":
                    ordered_types.append("e190")

        for j in range(len(ordered_types)):
            current_aircraft = ordered_types[j]
            occupied_step = sum(state_step)
            free_slot = h - occupied_step
            if free_slot > 0:
                if current_aircraft == "e190":
                    state_step[1] = state_step[1] + 1   
                    total_accepted_e190 = total_accepted_e190 + 1
                elif current_aircraft == "737":
                    state_step[2] = state_step[2] + 1   
                    total_accepted_737 = total_accepted_737 + 1
            else:
                if current_aircraft == "e190":
                    total_rejected_e190 = total_rejected_e190 + 1
                elif current_aircraft == "737":
                    total_rejected_737 = total_rejected_737 + 1

        state = state_step.copy() #One beat passed
        if sum(state) == h:
            full_bay_counter = full_bay_counter + 1
        total_beats = total_beats + 1

    #Totals per policy
    total_accepted = total_accepted_e190 + total_accepted_737
    total_rejected = total_rejected_e190 + total_rejected_737
    total_arrivals = total_accepted + total_rejected
    if total_arrivals > 0:
        general_rejection_rate = total_rejected / total_arrivals
    else:
        general_rejection_rate = 0
    
    NRV = (total_rejected_e190 * e190_net_revenue +
                      total_rejected_737 * b737_net_revenue)

    policy_results.append({"Policy": policy,
        "Arrivals": total_arrivals,
        "Accepted": total_accepted,
        "Rejected": total_rejected,
        "Rej e190": total_rejected_e190,
        "Rej 737": total_rejected_737,
        "Rej rate %": round(general_rejection_rate * 100, 4),
        "Full beats": full_bay_counter,
        "NRV/CO": round(NRV, 2)})

comparison_table = pd.DataFrame(policy_results)
print("\nPolicy comparison (same real arrivals, three paralell rules running):")
print(comparison_table.to_string(index=False))


Policy comparison (same real arrivals, three paralell rules running):
Policy  Arrivals  Accepted  Rejected  Rej e190  Rej 737  Rej rate %  Full beats  NRV/CO
  fifs      1688      1688         0         0        0         0.0           9     0.0
  e190      1688      1688         0         0        0         0.0           9     0.0
  b737      1688      1688         0         0        0         0.0           9     0.0
